# DiffuChat — Colab run (GPU preset)

Miniature masked-diffusion language model (LLaDA/MDLM style) trained from
scratch on TinyShakespeare, with the editable-answer chat primitive.

This notebook is runnable top-to-bottom on a free Colab T4:
1. clone + install the repo,
2. train the **diffusion LM** (gpu preset) and the matched **AR baseline**,
3. run the **3 experiments** (masked-span reconstruction vs baselines,
    decoding ablation, edit conditioning),
4. render a generation as a **frame animation** (denoising canvas over time),
5. save all artifacts for download.

In [ ]:
# 1. setup -----------------------------------------------------------------
# Option A: clone the repo (fill in your fork URL) -------------------------
# !git clone https://github.com/<you>/diffuchat.git
# %cd diffuchat
# Option B: upload the diffuchat/ folder, then:
import sys, os
if os.path.isdir('src/diffuchat'):
    %pip install -q -e .
else:
    print('clone or upload the diffuchat/ repo folder first')

%pip install -q matplotlib
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'

In [ ]:
# 2. data + tokenizer sanity ----------------------------------------------
from diffuchat.data import shakespeare, CharTokenizer, encode_corpus

text = shakespeare()                 # downloads TinyShakespeare if needed
tok = CharTokenizer(text)
ids = encode_corpus(tok, text)
print(f'corpus: {len(text):,} chars | vocab {tok.vocab_size} '
      f'(65 chars + [MASK] id {tok.mask_token_id})')
assert tok.mask_token_id == tok.vocab_size - 1 == 65
assert tok.mask_token_id not in set(ids.tolist())
print('round-trip OK:', tok.decode(tok.encode(text[:100])) == text[:100])

In [ ]:
# 3. train the diffusion LM (gpu preset: 384/12/8, 8000 steps) ------------
# ~25-35 min on a T4. For a quick pass set QUICK=True (cpu-ish preset).
QUICK = False
from diffuchat.config import get_preset
from diffuchat.train import train

mcfg, tcfg = get_preset('gpu' if not QUICK else 'cpu')
if QUICK:
    tcfg.steps = 600
tcfg.device = 'cuda'
out = train(mcfg, tcfg, 'artifacts/diffusion_lm_gpu.pt')
print('best val loss:', out['best_val_loss'])

In [ ]:
# 4. train the matched AR baseline (same shape, same steps, same data) ----
from diffuchat.ar_baseline import train_ar
from diffuchat.config import DenoiseLMConfig, TrainConfig

arcfg = DenoiseLMConfig(**{k: v for k, v in vars(mcfg).items()})
artcfg = TrainConfig(steps=tcfg.steps, batch_size=tcfg.batch_size, lr=tcfg.lr,
                     warmup=tcfg.warmup, eval_every=tcfg.eval_every,
                     device='cuda')
arout = train_ar(arcfg, artcfg, 'artifacts/ar_baseline_gpu.pt')
print('AR best val loss:', arout['best_val_loss'])

In [ ]:
# 5. generate + watch the canvas denoise ----------------------------------
from diffuchat.train import load_model
from diffuchat.generate import generate
from diffuchat.config import GenConfig
from diffuchat.data import chat_prefix

model, tok, ck = load_model('artifacts/diffusion_lm_gpu.pt')
model = model.to('cuda')
gcfg = GenConfig(strategy='confidence', steps=20, answer_len=96,
                 temperature=0.0, seed=42)
res = generate(model, tok, chat_prefix('The king said'), gcfg)
print('answer :', repr(res.answer_text))
print(f'{res.n_forward_passes} forward passes, {res.wall_time_s:.2f}s, '
      f'{len(res.frames)-1} frames')
for i in [0, len(res.frames)//3, 2*len(res.frames)//3, len(res.frames)-1]:
    frame = res.frames[i][res.canvas.answer_start:]
    print(f'frame {i:>3}: {tok.decode(frame)}')

In [ ]:
# 6. frame ANIMATION (denoising canvas as an imshow grid) -----------------
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML

start = res.canvas.answer_start
L = res.canvas.answer_len
cols = 32
rows = -(-L // cols)

def grid(ids):
    ans = ids[start:start + L]
    g = np.full((rows, cols), np.nan)
    for i, t in enumerate(ans):
        g[i // cols, i % cols] = t
    return g

fig, ax = plt.subplots(figsize=(10, 0.5 * rows + 1))
frames = []
for f in res.frames:
    g = grid(f)
    im = ax.imshow(g, cmap='tab20', vmin=0, vmax=66, aspect='auto')
    mask_overlay = ax.imshow(np.ma.masked_invalid(np.where(g == 65, 1.0, np.nan)),
                             cmap='gray', vmin=0, vmax=1, aspect='auto')
    txt = ax.set_title('')
    frames.append([im, mask_overlay, txt])
ax.set_axis_off()
anim = ArtistAnimation(fig, frames, interval=150, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())   # tokens pop in in ANY order — not left-to-right!

In [ ]:
# 7. the product primitive: EDIT -> re-denoise ----------------------------
from diffuchat.generate import edit

canvas = res.canvas
pos = canvas.answer_start + canvas.answer_len // 2
res2 = edit(model, tok, canvas, [(pos, tok.stoi['q'])],
            GenConfig(strategy='confidence', steps=20, answer_len=96,
                      temperature=0.0, seed=43))
assert res2.canvas.ids[pos] == tok.stoi['q'], 'pin must survive (§7.4)'
changed = sum(1 for i in range(canvas.answer_start, len(canvas.ids))
              if res2.canvas.ids[i] != res.frames[-1][i])
print('after edit  :', repr(res2.answer_text))
print(f'pin survived; {changed} answer tokens re-flowed around it')

In [ ]:
# 8. the 3 experiments -----------------------------------------------------
from diffuchat.evaluate import evaluate

report = evaluate(ckpt='artifacts/diffusion_lm_gpu.pt',
                  ar_ckpt='artifacts/ar_baseline_gpu.pt',
                  out='artifacts/eval_report_gpu.json')

In [ ]:
# 9. interactive-ish demo (all three strategies) ---------------------------
for strat in ('confidence', 'random', 'semi_ar'):
    r = generate(model, tok, chat_prefix('What news from Rome'),
                 GenConfig(strategy=strat, steps=20, answer_len=96,
                           temperature=0.0, seed=0))
    print(f'[{strat:>10}] {r.answer_text!r}  ({r.n_forward_passes} passes)')

In [ ]:
# 10. save artifacts for download -----------------------------------------
from google.colab import files
for f in ['artifacts/diffusion_lm_gpu.pt', 'artifacts/ar_baseline_gpu.pt',
          'artifacts/eval_report_gpu.json', 'artifacts/eval_report_gpu.md']:
    files.download(f)